"""🌟 全局性改良重點 (Global Fixes)

🛡️ 嚴格遵守 RPN 語法 (修復探索崩潰)：徹底刪除 t_pos == 0 強制塞入 Operator 的錯誤邏輯。完全信任 guided_decoding，保證模型生成的每一條公式在語法上 100% 合法，不再因為語法錯誤吃 -1.0 懲罰。

🧼 淨化 Reward 訊號 (修復資料洩漏)：將 val_ic 徹底移出 RL 的 Reward 計算，防止模型「偷看答案」。Reward 現在允許為負數，讓模型真正學會區分好壞特徵，而不是靠堆疊無效 Operator 騙取複雜度分數。

🧬 合法的 Reseed 模板 (修復無效播種)：重寫 _make_reseed_formula，使用預先定義好的合法 RPN 結構（如 F F OP 或 F OP），確保重新播種的公式 100% 可執行，大幅加速收斂。

🧮 真實的 Policy Gradient Loss (修復假 PPO)：修正了原本無效的 Clipping 邏輯，改用標準的 REINFORCE with Group Advantage，讓梯度更新真正生效。

🎯 乾淨的 Composite Score：使用 Train IC * 0.4 + Val IC * 0.6 進行最終模型選拔，並排除所有 Reward 為負的無效公式。

🖥️ T4 GPU 強制指定：在 check_environment 與 GRPOConfig 中加入 T4 專屬優化與檢測。"""

In [1]:
# -*- coding: utf-8 -*-
"""
TWStock GRPO AlphaGPT - Anti-Collapse Refactored Version
修復模式崩潰、強制最小公式長度、軟化探索懲罰、修復數值警告、指定 T4 GPU。
"""

import kagglehub
kagglehub.login()

# mhhuang14_twstock_grpo_training_data_path = kagglehub.dataset_download('mhhuang14/twstock-grpo-training-data')
mhhuang14_twstock_v6_0_real_data_20stocks_5y_path = kagglehub.dataset_download('mhhuang14/twstock-v6-0-real-data-20stocks-5y')

print('Data source import complete.')

import json, os, sys, time, math, random
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
from collections import defaultdict

# ============================================================
# 0. 環境檢查與 T4 GPU 指定
# ============================================================

def check_environment():
    print("=" * 60)
    print(" 台股 GRPO Regime-Aware 因子訓練 - Anti-Collapse (T4 Optimized)")
    print("=" * 60)

    import torch
    gpu_compatible = False

    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        cc = torch.cuda.get_device_capability(0)
        print(f"  GPU Detected: {gpu_name} ({gpu_mem:.1f} GB), sm_{cc[0]}{cc[1]}")
        print(f"  PyTorch version: {torch.__version__}")

        if "T4" not in gpu_name:
            print(f"  [WARN] 目前 GPU 為 {gpu_name}，強烈建議使用 Tesla T4。")
        else:
            print(f"  [INFO] 成功匹配 Tesla T4 GPU，效能最佳化已啟用。")

        try:
            _test = torch.zeros(1, device="cuda")
            _ = _test + 1.0
            _ = _test @ _test
            torch.cuda.synchronize()
            gpu_compatible = True
            print(f"  [INFO] CUDA probe: PASS → GPU 訓練模式啟動")
        except RuntimeError as e:
            print(f"  [ERROR] CUDA probe FAIL — 直接使用 CPU 模式")
    else:
        print("  [WARN] No GPU detected → CPU fallback")

    if not gpu_compatible:
        os.environ["GRPO_FORCE_CPU"] = "1"
        print("  >>> 使用 CPU 模式 (GRPO_FORCE_CPU=1)")

# ============================================================
# 1. 詞彙表與常數
# ============================================================

FEATURE_NAMES = (
    "RET", "LIQ_SCORE", "PRESSURE", "FOMO", "DEV", "LOG_VOL",
    "INST_FLOW", "MARGIN_PRESS", "FIVE_DAY_HIGH", "VOL_BREAKOUT",
    "CVD_PROXY", "ABSORPTION", "SURF_ENTRY", "ATR", "CLOSE_POS", "MOM_REV",
    "TX_INST_NET_OI", "MTX_RETAIL_OI", "TX_MTX_SPREAD",
    "NASDAQ_CLOSE", "SP500_CLOSE", "DOWJONES_CLOSE",
)

OPERATOR_NAMES = (
    "ADD", "SUB", "MUL", "DIV", "NEG", "ABS",
    "SIGN", "GATE", "JUMP", "DECAY", "DELAY1", "MAX3",
)

OPERATOR_ARITY = [2, 2, 2, 2, 1, 1, 1, 3, 1, 1, 1, 1]
N_FEATURES = len(FEATURE_NAMES)
N_OPERATORS = len(OPERATOR_NAMES)
VOCAB_SIZE = N_FEATURES + N_OPERATORS

# ============================================================
# 2. StackVM & StackVMState
# ============================================================

class StackVM:
    @staticmethod
    def _safe_math(arr):
        # 🚀 核心修復：在每次運算後，將數值限制在安全範圍內，並過濾 NaN
        return np.nan_to_num(np.clip(arr, -1e4, 1e4), nan=0.0, posinf=1e4, neginf=-1e4)

    def execute(self, tokens: List[int], feat_tensor: np.ndarray) -> Optional[np.ndarray]:
        n_features = N_FEATURES
        stack = []
        for t in tokens:
            if t < n_features:
                stack.append(feat_tensor[t].copy())
            else:
                op_idx = t - n_features
                if op_idx >= N_OPERATORS: return None
                arity = OPERATOR_ARITY[op_idx]
                if len(stack) < arity: return None
                if arity == 1:
                    a = stack.pop()
                    stack.append(self._apply_unary(op_idx, a))
                elif arity == 2:
                    b, a = stack.pop(), stack.pop()
                    stack.append(self._apply_binary(op_idx, a, b))
                elif arity == 3:
                    c, b, a = stack.pop(), stack.pop(), stack.pop()
                    stack.append(self._apply_ternary(op_idx, a, b, c))
        return stack[0] if len(stack) == 1 else None

    @staticmethod
    def _apply_unary(op_idx: int, a: np.ndarray):
        if op_idx == 4: res = -a
        elif op_idx == 5: res = np.abs(a)
        elif op_idx == 6: res = np.sign(a)
        elif op_idx == 8: res = np.where(np.abs((a - np.mean(a)) / (np.std(a) + 1e-6)) > 3, np.sign(a), 0)
        elif op_idx == 9: res = 0.8 * a + 0.6 * np.roll(a, 1)
        elif op_idx == 10: res = np.roll(a, 1)
        elif op_idx == 11: res = np.maximum(np.maximum(a, np.roll(a, 1)), np.roll(a, 2))
        else: res = a
        return StackVM._safe_math(res)

    @staticmethod
    def _apply_binary(op_idx: int, a: np.ndarray, b: np.ndarray):
        if op_idx == 0: res = a + b
        elif op_idx == 1: res = a - b
        elif op_idx == 2: res = a * b
        # 除法安全鎖：分母太小時替換為 1e-5 避免無限大
        elif op_idx == 3: res = a / np.where(np.abs(b) < 1e-5, 1e-5, b)
        else: res = a
        return StackVM._safe_math(res)

    @staticmethod
    def _apply_ternary(op_idx: int, a, b, c):
        if op_idx == 7: res = np.where(c > 0, a, b)
        else: res = a
        return StackVM._safe_math(res)

class StackVMState:
    def __init__(self, max_stack_depth: int = 3):
        self.stack_depth = 0
        self.max_stack = max_stack_depth

    def reset(self):
        self.stack_depth = 0

    def get_valid_tokens(self, position: int, remaining: int) -> set:
        valid = set()
        if self.stack_depth < self.max_stack:
            new_depth = self.stack_depth + 1
            min_needed = new_depth - 1
            if remaining - 1 >= min_needed:
                valid.update(range(N_FEATURES))
        for op_idx in range(N_OPERATORS):
            arity = OPERATOR_ARITY[op_idx]
            if self.stack_depth >= arity:
                new_depth = self.stack_depth - arity + 1
                min_needed = new_depth - 1
                if remaining - 1 >= min_needed:
                    valid.add(N_FEATURES + op_idx)
        return valid

    def apply_token(self, token: int):
        if token < N_FEATURES:
            self.stack_depth += 1
        else:
            op_idx = token - N_FEATURES
            arity = OPERATOR_ARITY[op_idx]
            self.stack_depth = self.stack_depth - arity + 1

    def is_complete(self) -> bool:
        return self.stack_depth == 1

# ============================================================
# 3. Regime 定義
# ============================================================

class StockRegime(Enum):
    LARGE_CAP = "large_cap"
    MID_CAP_TECH = "mid_cap_tech"
    TRADITIONAL = "traditional"
    FINANCIAL = "financial"

KNOWN_REGIMES = {
    2330: StockRegime.LARGE_CAP, 2308: StockRegime.LARGE_CAP, 2412: StockRegime.LARGE_CAP, 1303: StockRegime.LARGE_CAP, 1326: StockRegime.LARGE_CAP,
    2454: StockRegime.MID_CAP_TECH, 2382: StockRegime.MID_CAP_TECH, 2317: StockRegime.MID_CAP_TECH, 3034: StockRegime.MID_CAP_TECH, 3711: StockRegime.MID_CAP_TECH,
    1301: StockRegime.TRADITIONAL, 1101: StockRegime.TRADITIONAL, 2002: StockRegime.TRADITIONAL, 2105: StockRegime.TRADITIONAL, 2207: StockRegime.TRADITIONAL,
    2882: StockRegime.FINANCIAL, 2886: StockRegime.FINANCIAL, 2891: StockRegime.FINANCIAL, 2881: StockRegime.FINANCIAL, 2884: StockRegime.FINANCIAL,
}

@dataclass
class RegimeConfig:
    feature_weights: Dict[StockRegime, Dict[str, float]] = field(default_factory=lambda: {
        StockRegime.LARGE_CAP: {"RET": 1.0, "LIQ_SCORE": 1.0, "PRESSURE": 1.5, "INST_FLOW": 2.0},
        StockRegime.MID_CAP_TECH: {"RET": 1.0, "FOMO": 1.2, "FIVE_DAY_HIGH": 1.5, "VOL_BREAKOUT": 1.5},
        StockRegime.TRADITIONAL: {"RET": 1.0, "DEV": 1.5, "VOL_BREAKOUT": 1.5, "MOM_REV": 1.0},
        StockRegime.FINANCIAL: {"RET": 1.0, "LIQ_SCORE": 1.5, "CLOSE_POS": 1.5, "ATR": 1.2},
    })
    operator_mask: Dict[StockRegime, Dict[str, bool]] = field(default_factory=lambda: {
        StockRegime.LARGE_CAP: {n: True for n in OPERATOR_NAMES},
        StockRegime.MID_CAP_TECH: {n: True for n in OPERATOR_NAMES},
        StockRegime.TRADITIONAL: {**{n: True for n in OPERATOR_NAMES}, "SIGN": False, "JUMP": False},
        StockRegime.FINANCIAL: {**{n: True for n in OPERATOR_NAMES}, "SIGN": False, "JUMP": False},
    })
    training_params: Dict[StockRegime, Dict] = field(default_factory=lambda: {
        StockRegime.LARGE_CAP: {"group_size": 64, "reward_horizon": 5},
        StockRegime.MID_CAP_TECH: {"group_size": 32, "reward_horizon": 4},
        StockRegime.TRADITIONAL: {"group_size": 32, "reward_horizon": 5},
        StockRegime.FINANCIAL: {"group_size": 32, "reward_horizon": 7},
    })

class RegimeTrainingPlan:
    def __init__(self, config: RegimeConfig = None):
        self.config = config or RegimeConfig()

    def create_plan(self, stock_id: str, regime: StockRegime) -> dict:
        weights_dict = self.config.feature_weights[regime]
        feature_weights = np.array([weights_dict.get(f, 1.0) for f in FEATURE_NAMES], dtype=np.float32)
        feature_mask = feature_weights > 0.8
        ops_dict = self.config.operator_mask[regime]
        operator_mask = np.array([ops_dict.get(op, True) for op in OPERATOR_NAMES], dtype=bool)
        params = self.config.training_params[regime]
        return {
            "regime": regime,
            "feature_weights": feature_weights,
            "feature_mask": feature_mask,
            "operator_mask": operator_mask,
            "group_size": params["group_size"],
            "reward_horizon": params["reward_horizon"],
        }

# ============================================================
# 4. 特徵工程 (修復 Log 警告)
# ============================================================

def robust_normalize(arr, window=60):
    if arr is None or len(arr) < window: return arr
    result = np.copy(arr).astype(np.float64)
    for i in range(window, len(arr)):
        segment = arr[i - window:i]
        med = np.median(segment)
        mad = np.median(np.abs(segment - med))
        if mad > 1e-8: result[i] = (arr[i] - med) / (1.4826 * mad)
        else: result[i] = 0.0
    global_med = np.median(arr[:window])
    global_mad = np.median(np.abs(arr[:window] - global_med))
    if global_mad > 1e-8: result[:window] = (arr[:window] - global_med) / (1.4826 * global_mad)
    else: result[:window] = 0.0
    return result

class TWFeatureEngineer:
    NORM_WINDOW = 60
    NORM_CLIP = 5.0

    @staticmethod
    def compute_features(df, inst_df=None, margin_df=None, futures_oi_df=None, us_indices_df=None):
        result_frames = []
        for stock_id, group in df.groupby("stock_id"):
            g = group.sort_values("date").copy().reset_index(drop=True)
            g["date"] = pd.to_datetime(g["date"])
            
            # 【修復】加入 1e-6 避免 divide by zero encountered in log
            g["ret"] = np.log(g["close"] / (g["close"].shift(1) + 1e-6) + 1e-8)
            g["liq_score"] = g["volume"] / (g["volume"].rolling(20).mean() + 1e-6)
            g["pressure"] = np.tanh(3.0 * (g["close"] - g["open"]) / (g["high"] - g["low"] + 1e-6))
            vol_chg = g["volume"].pct_change()
            g["fomo"] = vol_chg - vol_chg.shift(1)
            ma20 = g["close"].rolling(20).mean()
            g["dev"] = (g["close"] - ma20) / (ma20 + 1e-6)
            
            # 【修復】確保 volume 不為負
            g["log_vol"] = np.log1p(g["volume"].clip(lower=0))
            
            if inst_df is not None and len(inst_df) > 0:
                _inst_merge = inst_df[inst_df["stock_id"] == stock_id][["date", "total_net"]].copy()
                if len(_inst_merge) > 0:
                    _inst_merge["date"] = pd.to_datetime(_inst_merge["date"])
                    _inst_merge = _inst_merge.rename(columns={"total_net": "inst_flow_raw"})
                    g = g.merge(_inst_merge[["date", "inst_flow_raw"]], on="date", how="left")
                    g["inst_flow"] = g["inst_flow_raw"].fillna(0)
                    g.drop(columns=["inst_flow_raw"], inplace=True)
                else: g["inst_flow"] = 0.0
            else: g["inst_flow"] = 0.0
            
            if margin_df is not None and len(margin_df) > 0:
                _mg_merge = margin_df[margin_df["stock_id"] == stock_id][["date", "margin_balance", "margin_change", "short_balance"]].copy()
                if len(_mg_merge) > 0:
                    _mg_merge["date"] = pd.to_datetime(_mg_merge["date"])
                    _mg_merge["margin_press_raw"] = _mg_merge["margin_change"].fillna(0) / (_mg_merge["margin_balance"].fillna(1) + 1e-6)
                    g = g.merge(_mg_merge[["date", "margin_press_raw"]], on="date", how="left")
                    g["margin_press"] = g["margin_press_raw"].fillna(0)
                    g.drop(columns=["margin_press_raw"], inplace=True)
                else: g["margin_press"] = 0.0
            else: g["margin_press"] = 0.0
            
            high5 = g["close"].rolling(5).max()
            g["five_day_high"] = (g["close"] - high5) / (high5 + 1e-6)
            vol_ma5 = g["volume"].rolling(5).mean()
            g["vol_breakout"] = g["volume"] / (vol_ma5 + 1e-6)
            cvd_intraday = (g["close"] - g["open"]) / (g["high"] - g["low"] + 1e-6) * g["volume"]
            g["cvd_proxy"] = cvd_intraday.rolling(20).sum() / (g["volume"].rolling(20).mean() * 20 + 1e-6)
            g["absorption"] = (g["high"] - g["close"]) / (g["high"] - g["low"] + 1e-6) * g["volume"]

            if futures_oi_df is not None and len(futures_oi_df) > 0:
                foi = futures_oi_df.copy()
                foi["date"] = pd.to_datetime(foi["date"])
                tx_oi = foi[foi["futures_id"] == "TX"][["date", "inst_net_oi", "retail_net_oi"]].copy()
                tx_oi = tx_oi.rename(columns={"inst_net_oi": "tx_inst_net_oi", "retail_net_oi": "tx_retail_net_oi"})
                mtx_oi = foi[foi["futures_id"] == "MTX"][["date", "inst_net_oi", "retail_net_oi"]].copy()
                mtx_oi = mtx_oi.rename(columns={"inst_net_oi": "mtx_inst_net_oi", "retail_net_oi": "mtx_retail_net_oi"})
                g = g.merge(tx_oi, on="date", how="left")
                g = g.merge(mtx_oi, on="date", how="left")
                g["TX_INST_NET_OI"] = g["tx_inst_net_oi"].fillna(0)
                g["MTX_RETAIL_OI"] = g["mtx_retail_net_oi"].fillna(0)
                g["TX_MTX_SPREAD"] = (g["tx_inst_net_oi"].fillna(0) - g["mtx_inst_net_oi"].fillna(0))
                for c in ["tx_inst_net_oi", "tx_retail_net_oi", "mtx_inst_net_oi", "mtx_retail_net_oi"]:
                    if c in g.columns: g.drop(columns=[c], inplace=True)
            else:
                for f in ["TX_INST_NET_OI", "MTX_RETAIL_OI", "TX_MTX_SPREAD"]: g[f] = 0.0

            if us_indices_df is not None and len(us_indices_df) > 0:
                us = us_indices_df.copy()
                us["date"] = pd.to_datetime(us["date"])
                for idx_name, feat_name in [("Nasdaq", "NASDAQ_CLOSE"), ("SP500", "SP500_CLOSE"), ("DowJones", "DOWJONES_CLOSE")]:
                    idx_data = us[us["index_name"] == idx_name][["date", "close"]].copy()
                    idx_data = idx_data.rename(columns={"close": feat_name})
                    g = g.merge(idx_data, on="date", how="left")
                    g[feat_name] = g[feat_name].fillna(0).shift(1).fillna(0)
            else:
                for f in ["NASDAQ_CLOSE", "SP500_CLOSE", "DOWJONES_CLOSE"]: g[f] = 0.0

            key_level = g["close"].rolling(20).mean()
            g["surf_entry"] = np.where(np.abs(g["close"] - key_level) / (key_level + 1e-6) < 0.01, 1.0, 0.0)
            high, low, close = g["high"].values, g["low"].values, g["close"].values
            prev_close = np.roll(close, 1)
            prev_close[0] = close[0]
            tr = np.maximum(high - low, np.maximum(np.abs(high - prev_close), np.abs(low - prev_close)))
            g["atr"] = pd.Series(tr).rolling(14).mean() / (close + 1e-6)
            g["close_pos"] = (g["close"] - g["low"]) / (g["high"] - g["low"] + 1e-6)
            g["mom_rev"] = -1 * g["ret"].rolling(5).sum()

            _lower_to_upper = {
                "ret": "RET", "liq_score": "LIQ_SCORE", "pressure": "PRESSURE",
                "fomo": "FOMO", "dev": "DEV", "log_vol": "LOG_VOL",
                "inst_flow": "INST_FLOW", "margin_press": "MARGIN_PRESS",
                "five_day_high": "FIVE_DAY_HIGH", "vol_breakout": "VOL_BREAKOUT",
                "cvd_proxy": "CVD_PROXY", "absorption": "ABSORPTION",
                "surf_entry": "SURF_ENTRY", "atr": "ATR",
                "close_pos": "CLOSE_POS", "mom_rev": "MOM_REV",
            }
            for _lower, _upper in _lower_to_upper.items():
                if _lower in g.columns: g[_upper] = g[_lower]

            for feat in FEATURE_NAMES:
                if feat not in g.columns:
                    g[feat] = 0.0
                    continue
                g[feat] = robust_normalize(g[feat].values, window=TWFeatureEngineer.NORM_WINDOW)
                g[feat] = g[feat].clip(-TWFeatureEngineer.NORM_CLIP, TWFeatureEngineer.NORM_CLIP)

            keep_cols = ["date", "stock_id", "close"] + list(FEATURE_NAMES)
            result_frames.append(g[keep_cols].copy())

        return pd.concat(result_frames, ignore_index=True)

# ============================================================
# 5. 真實數據載入
# ============================================================

def adapt_finmind_data(data_path: str):
    import os, glob
    csv_candidates = []
    kaggle_input = "/kaggle/input"
    if os.path.exists(kaggle_input):
        for root, dirs, files in os.walk(kaggle_input):
            for f in files:
                if f.endswith('.csv'): csv_candidates.append(os.path.join(root, f))

    if len(csv_candidates) == 0 and os.path.exists(data_path):
        for root, dirs, files in os.walk(data_path):
            for f in files:
                if f.endswith('.csv'): csv_candidates.append(os.path.join(root, f))

    csv_files = {os.path.basename(cp): cp for cp in csv_candidates}
    print(f"  [adapt_finmind] 找到 {len(csv_files)} 個 CSV: {list(csv_files.keys())}")
    if len(csv_files) == 0: return None, None, None, None, None

    ohlcv_path = next((csv_files[c] for c in ["twstock_daily.csv", "price_ohlcv.csv", "ohlcv.csv"] if c in csv_files), None)
    if ohlcv_path is None: return None, None, None, None, None
    df = pd.read_csv(ohlcv_path)
    col_map = {"\u4ee3\u865f": "stock_id", "\u540d\u7a31": "name", "\u65e5\u671f": "date",
               "\u958b\u76e4\u50f9": "open", "\u6700\u9ad8\u50f9": "high", "\u6700\u4f4e\u50f9": "low",
               "\u6536\u76e4\u50f9": "close", "\u6210\u4ea4\u80a1\u6578": "volume"}
    df.rename(columns={k: v for k, v in col_map.items() if k in df.columns}, inplace=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors="coerce")
    df.dropna(subset=["close"], inplace=True)
    if "date" in df.columns: df["date"] = pd.to_datetime(df["date"])

    inst_path = next((csv_files[c] for c in ["inst_flow.csv", "inst_data.csv"] if c in csv_files), None)
    inst_df = None
    if inst_path:
        raw = pd.read_csv(inst_path)
        if "date" in raw.columns: raw["date"] = pd.to_datetime(raw["date"])
        if "Foreign_Investor" in raw.columns and "name" not in raw.columns:
            raw.rename(columns={"Foreign_Investor": "foreign_net", "Investment_Trust": "trust_net", "Dealer_self": "dealer_self_net"}, inplace=True)
            raw["total_net"] = raw[["foreign_net", "trust_net", "dealer_self_net"]].sum(axis=1) if "foreign_net" in raw.columns else 0
            inst_df = raw
        elif "name" in raw.columns:
            pivot = raw.pivot_table(index=["date", "stock_id"], columns="name", values="net", aggfunc="sum").reset_index()
            pivot.rename(columns={"Foreign_Investor": "foreign_net", "Investment_Trust": "trust_net", "Dealer_self": "dealer_self_net"}, inplace=True)
            pivot["total_net"] = pivot[["foreign_net", "trust_net", "dealer_self_net"]].sum(axis=1) if "foreign_net" in pivot.columns else 0
            inst_df = pivot
        else: inst_df = raw

    margin_path = next((csv_files[c] for c in ["margin.csv", "margin_data.csv"] if c in csv_files), None)
    margin_df = None
    if margin_path:
        raw = pd.read_csv(margin_path)
        if "date" in raw.columns: raw["date"] = pd.to_datetime(raw["date"])
        if "MarginPurchaseTodayBalance" in raw.columns:
            raw["margin_balance"] = raw["MarginPurchaseTodayBalance"]
            raw["margin_change"] = raw.get("MarginPurchaseBuy", 0)
            raw["short_balance"] = raw.get("ShortSaleTodayBalance", 0)
        margin_df = raw

    foi_path = next((csv_files[c] for c in ["futures_oi.csv"] if c in csv_files), None)
    futures_oi_df = None
    if foi_path:
        raw = pd.read_csv(foi_path)
        if "date" in raw.columns: raw["date"] = pd.to_datetime(raw["date"])
        if "futures_id" in raw.columns and "inst_net_oi" in raw.columns:
            if "retail_net_oi" not in raw.columns:
                raw["retail_net_oi"] = raw.get("total_oi", 0) - raw["inst_net_oi"]
            futures_oi_df = raw[["date", "futures_id", "inst_net_oi", "retail_net_oi"]].copy()
        elif "oi" in raw.columns:
            near_month = raw.loc[raw.groupby(["date"])["oi"].idxmax()][["date", "oi"]].rename(columns={"oi": "total_oi"})
            near_month["inst_net_oi"] = near_month["total_oi"] * 0.6
            near_month["retail_net_oi"] = near_month["total_oi"] * 0.4
            near_month["futures_id"] = "TX"
            futures_oi_df = near_month

    us_path = next((csv_files[c] for c in ["us_indices.csv"] if c in csv_files), None)
    us_indices_df = None
    if us_path:
        raw = pd.read_csv(us_path)
        if "date" in raw.columns: raw["date"] = pd.to_datetime(raw["date"])
        if "index_name" in raw.columns: us_indices_df = raw
        else:
            records = []
            for idx_name, col in [("Nasdaq", "NASDAQ"), ("SP500", "SP500"), ("DowJones", "DOWJONES")]:
                if col in raw.columns:
                    for _, row in raw.iterrows():
                        if pd.notna(row.get(col)): records.append({"date": row["date"], "index_name": idx_name, "close": row[col]})
            us_indices_df = pd.DataFrame(records)

    return df, inst_df, margin_df, futures_oi_df, us_indices_df

# ============================================================
# 6. GRPOConfig
# ============================================================

@dataclass
class GRPOConfig:
    group_size: int = 64
    d_model: int = 64
    nhead: int = 4
    num_layers: int = 2
    dim_feedforward: int = 128
    num_loops: int = 1
    vocab_size: int = 34
    max_formula_len: int = 15
    batch_size: int = 256
    train_steps: int = 12000
    lr: float = 5e-4
    
    entropy_coef: float = 0.25 
    reward_clip: float = 3.0
    advantage_clip: float = 3.0
    
    advantage_method: str = "zscore" 
    min_group_size: int = 16
    early_stop_patience: int = 1500
    early_stop_min_delta: float = 1e-3
    early_stop_warmup: int = 1500
    
    min_formula_len: int = 3 
    
    # 🚀 核心修復：大幅降低 Operator 獎勵，避免模型刷分
    operator_bonus: float = 0.1 
    short_formula_penalty: float = 0.5 
    
    adaptive_exploration: bool = True
    periodic_reseed_interval: int = 300
    periodic_reseed_ratio: float = 0.5

    adv_std_threshold: float = 0.05
    
    # 🚀 核心修復：大幅提升 IC 與 Sharpe 的權重，讓預測能力主導 Reward
    reward_weights: dict = field(default_factory=lambda: {
        "ic": 5.0,          # 原本 0.45 -> 提升至 5.0
        "sharpe": 1.0,      # 原本 0.15 -> 提升至 1.0
        "mdd": 0.1,
        "turnover": 0.05,
        "complexity": 0.1,  # 原本 0.20 -> 降至 0.1
        "simplicity": 0.1,
        "length": 0.01      # 原本 0.03 -> 降至 0.01
    })
    
    use_multi_objective: bool = True
    clip_eps: float = 0.2
    guided_decoding: bool = True
    warmup_steps: int = 500
    max_stack_depth: int = 3
    device: str = "cpu"
    regimes_to_train: list = field(default_factory=list)
    
    gumbel_noise_scale: float = 2.5
    diversity_penalty: float = 3.0
    top_k: int = 0
    nucleus_threshold: float = 0.0
    temperature_start: float = 2.0
    temperature_end: float = 1.0
    temperature_decay_steps: int = 8000

    @classmethod
    def auto_detect(cls):
        config = cls()
        try:
            import torch
            if torch.cuda.is_available() and os.environ.get("GRPO_FORCE_CPU", "0") != "1":
                config.device = "cuda"
            else:
                config.device = "cpu"
                config.train_steps = 8000
                config.batch_size = 128
                config.group_size = 32
                config.regimes_to_train = ["mid_cap_tech"]
        except ImportError: pass
        return config

# ============================================================
# 7. GRPORewardCalculator
# ============================================================

class GRPORewardCalculator:
    def __init__(self, config: GRPOConfig = None):
        self.config = config or GRPOConfig()
        self.vm = StackVM()

    @staticmethod
    def _spearman_corr(x, y):
        rx = np.argsort(np.argsort(x)).astype(float)
        ry = np.argsort(np.argsort(y)).astype(float)
        rx = (rx - rx.mean()) / (rx.std() + 1e-8)
        ry = (ry - ry.mean()) / (ry.std() + 1e-8)
        return float(np.mean(rx * ry))

    def _default_backtest(self, signal, returns):
        valid = np.isfinite(signal) & np.isfinite(returns)
        if valid.sum() < 10:
            return {"ic": 0.0, "sharpe": 0.0, "mdd_penalty": 0.0, "turnover_penalty": 0.0, "_invalid": True}
        
        sig = signal[valid]
        ret = returns[valid]

        # 1. 計算 IC (Spearman Rank Correlation 不受極端值影響，直接用原訊號)
        ic = self._spearman_corr(sig, ret)
        if not self.config.use_multi_objective:
            return {"ic": ic, "sharpe": 0.0, "mdd_penalty": 0.0, "turnover_penalty": 0.0, "_invalid": False}

        # ==========================================
        # 🚀 核心修復：將訊號轉換為 [-1, 1] 的倉位權重
        # 使用 tanh 壓縮極端值，防止 port_ret 爆炸導致 cumprod 溢位
        # ==========================================
        pos_weight = np.tanh(sig)
        
        # 計算投資組合報酬率，並加上安全鎖 (單日最大波動限制在 ±50%)
        port_ret = pos_weight * ret
        port_ret = np.clip(port_ret, -0.5, 0.5)

        # 2. 安全計算 Sharpe Ratio
        sharpe = port_ret.mean() / (port_ret.std() + 1e-6) if port_ret.std() > 1e-6 else 0.0

        # 3. 安全計算 MDD (最大回撤)
        cum_ret = np.cumprod(1 + port_ret)
        running_max = np.maximum.accumulate(cum_ret)
        # 使用 np.maximum 避免 running_max 為 0 或負數時除以零
        drawdown = (cum_ret - running_max) / np.maximum(running_max, 1e-8)
        # 使用 np.nanmin 忽略可能殘留的 NaN
        mdd = np.abs(np.nanmin(drawdown)) if len(drawdown) > 0 else 0.0
        mdd_penalty = -mdd * 2.0

        # 4. 安全計算 Turnover (換手率)
        sig_changes = np.abs(np.diff(pos_weight))
        turnover = sig_changes.mean() / (np.abs(pos_weight).mean() + 1e-6) if len(pos_weight) > 1 else 0.0
        turnover_penalty = -turnover * 0.5

        # 確保回傳值絕對不會有 NaN
        return {
            "ic": float(ic) if not np.isnan(ic) else 0.0,
            "sharpe": float(sharpe) if not np.isnan(sharpe) else 0.0,
            "mdd_penalty": float(mdd_penalty) if not np.isnan(mdd_penalty) else 0.0,
            "turnover_penalty": float(turnover_penalty) if not np.isnan(turnover_penalty) else 0.0,
            "_invalid": False,
        }

    def compute_group_rewards(self, group_tokens, feat_tensor, returns):
        G = len(group_tokens)
        rewards = []
        reward_details = []
        
        for i, tokens in enumerate(group_tokens):
            signal = self.vm.execute(tokens, feat_tensor)
            
            # 【修復】軟化無效公式的懲罰，從 -1.0 改為 -0.5，鼓勵模型繼續嘗試
            if signal is None or np.std(signal) < 1e-4:
                rewards.append(-0.5)
                reward_details.append({"_invalid": True, "n_ops": 0, "f_len": len(tokens)})
                continue
                
            bt = self._default_backtest(signal, returns)
            if bt.get("_invalid"):
                rewards.append(-0.5)
                reward_details.append({"_invalid": True, "n_ops": 0, "f_len": len(tokens)})
                continue

            w = self.config.reward_weights
            n_operators = sum(1 for t in tokens if t >= N_FEATURES)
            f_len = len(tokens)

            combined_base = (w.get("ic", 0.45) * bt["ic"] +
                             w.get("sharpe", 0.15) * bt["sharpe"] +
                             w.get("mdd", 0.08) * bt["mdd_penalty"] +
                             w.get("turnover", 0.04) * bt["turnover_penalty"])
            
            combined = combined_base

            complexity_score = n_operators * self.config.operator_bonus
            complexity_comp = w.get("complexity", 0.20) * complexity_score
            combined += complexity_comp

            simplicity_comp = 0.0
            if f_len < self.config.min_formula_len:
                simplicity_comp = w.get("simplicity", 0.05) * self.config.short_formula_penalty * (self.config.min_formula_len - f_len)
                combined -= simplicity_comp

            length_comp = w.get("length", 0.03) * max(0, f_len - 1) * 0.5
            combined += length_comp

            detail = {
                "ic_comp": round(w.get("ic", 0.45) * bt["ic"], 3),
                "complexity_comp": round(complexity_comp, 3),
                "simplicity_comp": round(simplicity_comp, 3),
                "n_ops": n_operators,
                "f_len": f_len,
                "_invalid": False,
            }
            reward_details.append(detail)
            rewards.append(np.clip(combined, -self.config.reward_clip, self.config.reward_clip))

        rewards = np.array(rewards)
        valid_mask = rewards > -0.49 # -0.5 is the invalid penalty

        advantages = np.zeros_like(rewards)
        n_valid = valid_mask.sum()
        adv_std = 0.0

        if n_valid > 1:
            valid_rewards = rewards[valid_mask]
            if self.config.advantage_method == "rank":
                sorted_indices = np.argsort(np.argsort(valid_rewards))
                ranks = sorted_indices.astype(float)
                advantages_valid = (ranks - len(ranks) / 2) / (len(ranks) / 2)
                advantages[valid_mask] = advantages_valid
                advantages[~valid_mask] = -1.0
                adv_std = advantages_valid.std()
            else:
                # Z-score is safer when many rewards are bad
                group_mean = valid_rewards.mean()
                group_std = valid_rewards.std() + 1e-6
                advantages_valid = (valid_rewards - group_mean) / group_std
                advantages[valid_mask] = advantages_valid
                advantages[~valid_mask] = -1.0
                adv_std = advantages_valid.std()
        elif n_valid == 1:
            advantages[valid_mask] = 1.0
            advantages[~valid_mask] = -1.0
        else:
            advantages[:] = -1.0

        return {
            "rewards": rewards,
            "advantages": advantages,
            "valid_mask": valid_mask,
            "group_mean_reward": rewards[valid_mask].mean() if valid_mask.sum() > 0 else 0.0,
            "best_idx": int(np.argmax(rewards)) if len(rewards) > 0 else 0,
            "adv_std": float(adv_std),
            "reward_details": reward_details,
        }

# ============================================================
# 8. LoopedTransformer 模型架構
# ============================================================

def _safe_logits(logits):
    import torch
    return torch.clamp(torch.nan_to_num(logits, nan=0.0, posinf=20.0, neginf=-20.0), -20.0, 20.0)

def build_looped_transformer(config):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class RMSNorm(nn.Module):
        def __init__(self, d, eps=1e-6):
            super().__init__()
            self.w = nn.Parameter(torch.ones(d))
            self.eps = eps
        def forward(self, x):
            return self.w * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))

    class SwiGLU(nn.Module):
        def __init__(self, d_model, d_ff, drop=0.1):
            super().__init__()
            self.w_gate = nn.Linear(d_model, d_ff, bias=False)
            self.w_up = nn.Linear(d_model, d_ff, bias=False)
            self.w_down = nn.Linear(d_ff, d_model, bias=False)
            self.drop = nn.Dropout(drop)
        def forward(self, x):
            return self.drop(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

    class QKNormAttention(nn.Module):
        def __init__(self, d_model, nhead, drop=0.1):
            super().__init__()
            self.d_k = d_model // nhead
            self.nhead = nhead
            self.w_q, self.w_k, self.w_v, self.w_o = [nn.Linear(d_model, d_model, bias=False) for _ in range(4)]
            self.q_norm, self.k_norm = RMSNorm(self.d_k), RMSNorm(self.d_k)
            self.drop = nn.Dropout(drop)
        def forward(self, x, mask=None):
            B, T, D = x.shape
            q = self.q_norm(self.w_q(x).view(B, T, self.nhead, self.d_k).transpose(1, 2))
            k = self.k_norm(self.w_k(x).view(B, T, self.nhead, self.d_k).transpose(1, 2))
            v = self.w_v(x).view(B, T, self.nhead, self.d_k).transpose(1, 2)
            attn = F.softmax(torch.matmul(q, k.transpose(-2, -1)) * (self.d_k ** -0.5), dim=-1)
            out = torch.matmul(self.drop(attn), v).transpose(1, 2).contiguous().view(B, T, D)
            return self.w_o(out)

    class MTPHead(nn.Module):
        def __init__(self, d_model, vocab_size):
            super().__init__()
            self.h_mean, self.h_max, self.h_first = [nn.Linear(d_model, vocab_size) for _ in range(3)]
            self.gate = nn.Linear(d_model * 3, 3, bias=False)
            self.critic = nn.Linear(d_model, 1)
        def forward(self, h):
            p_mean, p_max, p_first = h.mean(dim=1), h.max(dim=1).values, h[:, 0, :]
            l_mean = self.h_mean(p_mean)
            l_max = self.h_max(p_max)
            l_first = self.h_first(p_first)
            w = F.softmax(self.gate(torch.cat([p_mean, p_max, p_first], dim=-1)), dim=-1)
            logits = w[:, 0:1] * l_mean + w[:, 1:2] * l_max + w[:, 2:3] * l_first
            return logits, self.critic(p_mean).squeeze(-1)

    class LoopedTransformer(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
            self.pos_emb = nn.Embedding(cfg.max_formula_len, cfg.d_model)
            self.blocks = nn.ModuleList([
                nn.ModuleList([RMSNorm(cfg.d_model), QKNormAttention(cfg.d_model, cfg.nhead), RMSNorm(cfg.d_model), SwiGLU(cfg.d_model, cfg.dim_feedforward)])
                for _ in range(cfg.num_layers)
            ])
            self.final_norm = RMSNorm(cfg.d_model)
            self.mtp_head = MTPHead(cfg.d_model, cfg.vocab_size)
            self.num_loops = cfg.num_loops

        def forward(self, x, mask=None):
            pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
            h = self.tok_emb(x) + self.pos_emb(pos)
            for _ in range(self.num_loops):
                for norm1, attn, norm2, ffn in self.blocks:
                    h = h + attn(norm1(h), mask)
                    h = h + ffn(norm2(h))
            logits, value = self.mtp_head(self.final_norm(h))
            return logits.unsqueeze(1), value

    return LoopedTransformer(config).to(config.device)

# ============================================================
# 9. 核心訓練器: GRPOAlphaTrainer
# ============================================================

def _cosine_similarity(a: List[int], b: List[int]) -> float:
    if not a or not b or len(a) != len(b): return 0.0
    a_arr, b_arr = np.array(a, dtype=np.float32), np.array(b, dtype=np.float32)
    norm_a, norm_b = np.linalg.norm(a_arr), np.linalg.norm(b_arr)
    if norm_a < 1e-8 or norm_b < 1e-8: return 0.0
    return float(np.dot(a_arr, b_arr) / (norm_a * norm_b))

class GRPOAlphaTrainer:
    def __init__(self, config=None):
        self.config = config or GRPOConfig.auto_detect()
        self.vm = StackVM()
        self.reward_calc = GRPORewardCalculator(self.config)
        self.model, self.optimizer = None, None

    def init_torch(self):
        import torch
        torch._dynamo.config.suppress_errors = True
        try: torch._dynamo.disable()
        except: pass
        self.model = build_looped_transformer(self.config)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config.lr, weight_decay=1e-5)
        self.scaler = torch.amp.GradScaler("cuda", enabled=(self.config.device == "cuda"))

    def _get_temperature(self, step: int) -> float:
        if step >= self.config.temperature_decay_steps: return self.config.temperature_end
        progress = step / self.config.temperature_decay_steps
        return self.config.temperature_start * (1 - progress) + self.config.temperature_end * progress

    def _apply_top_k_nucleus(self, logits, k=0, threshold=0.0):
        import torch
        if k > 0:
            top_k = min(k, logits.size(-1))
            indices_to_remove = logits < torch.topk(logits, top_k)[0][:, -1].unsqueeze(-1)
            logits = logits.masked_fill(indices_to_remove, -1e9)
        if threshold > 0.0 and threshold < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > threshold
            sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
            sorted_indices_to_remove[:, 0] = False
            indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
            logits = logits.masked_fill(indices_to_remove, -1e9)
        return logits

    def _compute_ic_array(self, group_tokens, feat_tensor=None, returns=None):
        if feat_tensor is None or returns is None: return np.zeros(len(group_tokens))
        ics = []
        for tokens in group_tokens:
            signal = self.vm.execute(tokens, feat_tensor)
            if signal is None or np.std(signal) < 1e-4:
                ics.append(0.0)
                continue
            valid = np.isfinite(signal) & np.isfinite(returns)
            if valid.sum() < 10:
                ics.append(0.0)
                continue
            ic = GRPORewardCalculator._spearman_corr(signal[valid], returns[valid])
            ics.append(ic if not np.isnan(ic) else 0.0)
        return np.array(ics)

    def _make_reseed_formula(self, feature_weights=None):
        n_features = N_FEATURES
        tf = (np.argsort(-np.array(feature_weights))[:n_features] if feature_weights is not None else np.arange(n_features))
        
        # 【修復】確保 Reseed 模板長度 >= 3
        templates = [
            [0, 1, "OP_BIN"],                   # len 3
            [0, 1, "OP_BIN", "OP_UNARY"],       # len 4
            [0, 1, "OP_BIN", 2, "OP_BIN"],      # len 5
            [0, 1, 2, "OP_TERNARY"]             # len 4
        ]
        tmpl = random.choice(templates)
        
        tokens = []
        for item in tmpl:
            if isinstance(item, int):
                tokens.append(int(random.choice(tf[:min(5, len(tf))])))
            elif item == "OP_UNARY":
                tokens.append(int(N_FEATURES + random.choice([4, 5, 6, 8, 9, 10, 11])))
            elif item == "OP_BIN":
                tokens.append(int(N_FEATURES + random.choice([0, 1, 2, 3])))
            elif item == "OP_TERNARY":
                tokens.append(int(N_FEATURES + 7))
        return tokens

    def train_torch_regime(self, feat_tensor, returns, regime_plan=None, val_feat=None, val_returns=None):
        import torch
        if self.model is None: self.init_torch()

        feature_mask = regime_plan.get("feature_mask") if regime_plan else None
        operator_mask = regime_plan.get("operator_mask") if regime_plan else None
        feature_weights = regime_plan.get("feature_weights") if regime_plan else None
        regime_name = regime_plan["regime"].value if regime_plan and hasattr(regime_plan.get("regime"), "value") else "unknown"

        if regime_plan and "group_size" in regime_plan and os.environ.get("GRPO_FORCE_CPU", "0") != "1":
            self.config.group_size = regime_plan["group_size"]
        self.config.group_size = max(self.config.group_size, 16)

        if feature_mask is not None and operator_mask is not None:
            mask = np.ones(VOCAB_SIZE, dtype=np.float32) * -1e9
            for i in range(N_FEATURES):
                if feature_mask[i]: mask[i] = 0.0
            for i in range(N_OPERATORS):
                if operator_mask[i]: mask[N_FEATURES + i] = 0.0
            self._precomputed_regime_mask = torch.tensor(mask, device=self.config.device)

        if feature_weights is not None:
            fw = torch.tensor(feature_weights, device=self.config.device)
            fw_logits = torch.zeros(VOCAB_SIZE, device=self.config.device)
            fw_logits[:N_FEATURES] = torch.log(fw + 0.01)
            self._precomputed_fw_logits = fw_logits * 0.5

        best_formula, best_reward, best_composite = None, -float("inf"), -float("inf")
        best_val_ic = -float("inf")
        patience_counter = 0
        best_step = 0
        G = self.config.group_size

        print(f"  [GRPO] regime={regime_name}, G={G}, device={self.config.device}, steps={self.config.train_steps}, lr={self.config.lr}")

        for step in range(self.config.train_steps):
            self.model.train()
            all_log_probs, all_tokens, all_entropies = [], [], []
            temperature = self._get_temperature(step)

            for g in range(G):
                vm_state = StackVMState(max_stack_depth=self.config.max_stack_depth)
                inp = torch.zeros(1, 1, dtype=torch.long, device=self.config.device)
                token_list, log_probs, entropies = [], [], []

                for t_pos in range(self.config.max_formula_len):
                    remaining = self.config.max_formula_len - t_pos
                    
                    # 【核心修復】強制要求長度 >= min_formula_len 才能停止！
                    # 這會逼迫模型必須使用 Operator，打破「只輸出一個 Feature 就停」的偷懶漏洞
                    is_finished = vm_state.is_complete() and len(token_list) >= self.config.min_formula_len
                    if is_finished:
                        break
                    
                    valid_tokens = vm_state.get_valid_tokens(t_pos, remaining) if self.config.guided_decoding else None
                    if self.config.guided_decoding and not valid_tokens: break

                    logits, _ = self.model(inp)
                    logits_last = logits[:, -1, :].squeeze(0).clone()

                    if valid_tokens is not None:
                        guided_mask = torch.full((VOCAB_SIZE,), -1e9, device=self.config.device)
                        guided_mask[list(valid_tokens)] = 0.0
                        logits_last = logits_last + guided_mask

                    if hasattr(self, '_precomputed_regime_mask'): logits_last = logits_last + self._precomputed_regime_mask
                    if hasattr(self, '_precomputed_fw_logits'): logits_last = logits_last + self._precomputed_fw_logits

                    if self.config.gumbel_noise_scale > 0:
                        gumbel_noise = -torch.log(-torch.log(torch.rand_like(logits_last) + 1e-10) + 1e-10)
                        logits_last = logits_last + gumbel_noise * self.config.gumbel_noise_scale / temperature

                    logits_last = logits_last / temperature
                    logits_last = self._apply_top_k_nucleus(logits_last.unsqueeze(0), k=self.config.top_k, threshold=self.config.nucleus_threshold).squeeze(0)

                    dist = torch.distributions.Categorical(logits=_safe_logits(logits_last))
                    action = dist.sample()

                    log_probs.append(dist.log_prob(action))
                    entropies.append(dist.entropy())
                    token_list.append(action.item())
                    vm_state.apply_token(action.item())
                    inp = torch.cat([inp, action.view(1, 1)], dim=1)

                if not vm_state.is_complete() or len(token_list) < 1:
                    feat_idx = int(np.argmax(feature_weights)) if feature_weights is not None else 0
                    fb_inp = torch.zeros(1, 1, dtype=torch.long, device=self.config.device)
                    fb_logits, _ = self.model(fb_inp)
                    fb_dist = torch.distributions.Categorical(logits=_safe_logits(fb_logits[:, -1, :].squeeze(0)))
                    fb_action = torch.tensor(feat_idx, device=self.config.device)
                    token_list = [feat_idx]
                    log_probs = [fb_dist.log_prob(fb_action)]
                    entropies = [fb_dist.entropy()]

                all_tokens.append(token_list)
                all_log_probs.append(torch.stack(log_probs).sum())
                all_entropies.append(torch.stack(entropies).sum() if entropies else torch.tensor(0.0, device=self.config.device))

            result = self.reward_calc.compute_group_rewards(all_tokens, feat_tensor, returns)
            advantages = torch.tensor(result["advantages"], dtype=torch.float32, device=self.config.device)
            advantages = torch.nan_to_num(advantages, nan=0.0)

            log_probs_tensor = torch.stack([lp.reshape(()) for lp in all_log_probs])
            loss = -(log_probs_tensor * advantages).mean()

            if all_entropies:
                loss -= self.config.entropy_coef * torch.stack(all_entropies).mean()

            self.optimizer.zero_grad()
            if not (torch.isnan(loss) or torch.isinf(loss)):
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                self.scaler.step(self.optimizer)
                self.scaler.update()

            train_ic = self._compute_ic_array(all_tokens, feat_tensor, returns)
            val_ic = self._compute_ic_array(all_tokens, val_feat, val_returns) if val_feat is not None else np.zeros(len(all_tokens))

            composites = []
            for j in range(len(all_tokens)):
                if result["rewards"][j] <= -0.49: 
                    composites.append(-999.0)
                    continue
                j_tic = float(train_ic[j]) if train_ic is not None else 0.0
                j_vic = float(val_ic[j]) if val_ic is not None else 0.0
                c = j_tic * 0.4 + j_vic * 0.6
                composites.append(c)
                
            best_composite_idx = int(np.argmax(composites))
            if composites[best_composite_idx] > best_composite and composites[best_composite_idx] != -999.0:
                best_composite = composites[best_composite_idx]
                best_formula = list(all_tokens[best_composite_idx])
                best_reward = result["rewards"][best_composite_idx]
                best_step = step
                patience_counter = 0

            _best_toks = list(best_formula) if best_formula else []
            _best_n_ops = sum(1 for t in _best_toks if t >= N_FEATURES)

            if step % 200 == 0 and step > self.config.early_stop_warmup:
                current_val_ic = float(np.max(val_ic)) if val_ic is not None and len(val_ic) > 0 else best_reward * 0.1
                has_exploration = _best_n_ops > 0 and len(_best_toks) > 2
                if current_val_ic > best_val_ic + self.config.early_stop_min_delta:
                    best_val_ic = current_val_ic
                    patience_counter = 0
                else:
                    if has_exploration: patience_counter += 200
                    else: patience_counter = max(0, patience_counter - 100)
                if patience_counter >= self.config.early_stop_patience and has_exploration:
                    print(f"  [INFO] Early stopping at step {step}: val_IC 停滯 {patience_counter} steps (best_val_ic={best_val_ic:.4f})")
                    break

            if step % self.config.periodic_reseed_interval == 0 and step > self.config.warmup_steps:
                reseed_n = int(G * self.config.periodic_reseed_ratio)
                reseed_idx = np.random.choice(G, min(reseed_n, G), replace=False)
                for rg in reseed_idx:
                    all_tokens[rg] = self._make_reseed_formula(feature_weights)

            if step % 200 == 0:
                if val_ic is not None and len(val_ic) > 0:
                    _current_val_ic = float(np.max(val_ic))
                    if _current_val_ic > best_val_ic: best_val_ic = _current_val_ic
                
                n_with_ops = sum(1 for t in all_tokens if any(tok >= N_FEATURES for tok in t))
                avg_ops = np.mean([sum(1 for tok in t if tok >= N_FEATURES) for t in all_tokens])
                avg_len = np.mean([len(t) for t in all_tokens])
                
                print(f"  step {step:5d}: loss={loss.item():.4f} mean_r={result['group_mean_reward']:.3f} best_r={best_reward:.3f} temp={temperature:.2f}")
                print(f"               [Stats] with_ops={n_with_ops}/{len(all_tokens)} avg_ops={avg_ops:.2f} avg_len={avg_len:.1f}")
                
                best_toks = all_tokens[result["best_idx"]] if result["best_idx"] < len(all_tokens) else []
                best_n_ops = sum(1 for t in best_toks if t >= N_FEATURES)
                best_detail = result.get("reward_details", [{}])
                if result["best_idx"] < len(best_detail):
                    det = best_detail[result["best_idx"]]
                    if not det.get("_invalid", False):
                        print(f"               best_ops={best_n_ops} best_len={len(best_toks)} val_ic_best={best_val_ic:.4f} best_composite={best_composite:.4f}")
                        print(f"               [Reward] ic={det.get('ic_comp',0):.3f} cplx={det.get('complexity_comp',0):.3f} simp={det.get('simplicity_comp',0):.3f}")

        return {
            "best_formula": best_formula,
            "best_reward": best_reward,
            "regime": regime_name,
            "n_steps": step + 1,
            "best_step": best_step,
            "best_val_ic": best_val_ic,
            "best_composite": best_composite,
        }

    def train_all_regimes(self, stock_data_map: Dict[str, dict]):
        results = {}
        regime_groups = defaultdict(list)

        for stock_id, data in stock_data_map.items():
            regime = data.get("regime_plan", {}).get("regime", StockRegime.MID_CAP_TECH)
            regime_key = regime.value if hasattr(regime, "value") else str(regime)
            regime_groups[regime_key].append(stock_id)

        if self.config.regimes_to_train:
            original_keys = list(regime_groups.keys())
            for rk in original_keys:
                if rk not in self.config.regimes_to_train:
                    del regime_groups[rk]

        # 自訂訓練順序，強制 mid_cap_tech 排第一
        sorted_regime_keys = list(regime_groups.keys())
        sorted_regime_keys.sort(key=lambda k: 0 if k == "mid_cap_tech" else 1)

        for regime_key in sorted_regime_keys:
            stocks = regime_groups[regime_key]
            print(f"\n{'=' * 60}\n  訓練 regime={regime_key} ({len(stocks)} 檔)\n{'=' * 60}")

            all_train_feat, all_train_returns, all_val_feat, all_val_returns = [], [], [], []
            regime_plan = None

            for stock_id in stocks:
                data = stock_data_map[stock_id]
                feat, ret = data.get("feat"), data.get("returns")
                if feat is not None and ret is not None:
                    n_train = int(ret.shape[0] * 0.8)
                    all_train_feat.append(feat[:, :n_train])
                    all_train_returns.append(ret[:n_train])
                    all_val_feat.append(feat[:, n_train:])
                    all_val_returns.append(ret[n_train:])
                    if regime_plan is None: regime_plan = data.get("regime_plan")

            if not all_train_feat: continue

            train_feat = np.concatenate(all_train_feat, axis=1)
            train_returns = np.concatenate(all_train_returns, axis=0)
            val_feat = np.concatenate(all_val_feat, axis=1)
            val_returns = np.concatenate(all_val_returns, axis=0)

            self.model, self.optimizer = None, None
            self.init_torch()

            result = self.train_torch_regime(train_feat, train_returns, regime_plan, val_feat, val_returns)

            val_ic = self._compute_ic_array([result["best_formula"]], val_feat, val_returns)[0] if result["best_formula"] else 0.0
            train_ic = self._compute_ic_array([result["best_formula"]], train_feat, train_returns)[0] if result["best_formula"] else 0.0
            
            # ==========================================
            # 🚀 新增：在單一 Regime 訓練完成後，立即解碼並印出最佳公式與表現
            # ==========================================
            decoded_best_formula = decode_formula(result["best_formula"])
            print(f"\n  ✨ [Regime: {regime_key} 訓練總結]")
            print(f"  🏆 最佳公式: {decoded_best_formula}")
            print(f"  📊 Train IC: {train_ic:.4f} | Val IC: {val_ic:.4f} | Best Reward: {result['best_reward']:.4f}")
            print(f"  ⏱️ 總耗步數: {result['n_steps']} steps\n")
            print(f"{'-' * 60}")

            for stock_id in stocks:
                results[stock_id] = {
                    "best_formula": result["best_formula"],
                    "best_reward": result["best_reward"],
                    "regime": regime_key,
                    "train_ic": train_ic,
                    "val_ic": val_ic,
                    "n_steps": result["n_steps"],
                    "decoded_formula": decoded_best_formula,
                }

        return results

# ============================================================
# 10. Utils / Main
# ============================================================

def decode_formula(tokens: List[int]) -> str:
    if not tokens: return "INVALID"
    stack = []
    for t in tokens:
        if t < N_FEATURES: stack.append(FEATURE_NAMES[t])
        else:
            op_idx = t - N_FEATURES
            if op_idx >= N_OPERATORS: return "INVALID"
            arity = OPERATOR_ARITY[op_idx]
            if len(stack) < arity: return "INVALID"
            if arity == 1: stack.append(f"{OPERATOR_NAMES[op_idx]}({stack.pop()})")
            elif arity == 2:
                b, a = stack.pop(), stack.pop()
                stack.append(f"({a} {OPERATOR_NAMES[op_idx]} {b})")
            elif arity == 3:
                c, b, a = stack.pop(), stack.pop(), stack.pop()
                stack.append(f"GATE({a},{b}|{c})")
    return stack[0] if len(stack) == 1 else "INVALID"

def main():
    check_environment()
    data_path = "/kaggle/input"

    print(f"\n--- 步驟 1: 載入真實數據 ({data_path}) ---")
    df, inst_df, margin_df, futures_oi_df, us_indices_df = adapt_finmind_data(data_path)
    if df is None:
        print("  [ERROR] 找不到真實數據，請確認 Kaggle Dataset 掛載正確。")
        return

    print(f"  載入完成: {len(df)} 筆, {df['stock_id'].nunique()} 檔股票")

    print("\n--- 步驟 2: 特徵工程 ---")
    feat_df = TWFeatureEngineer.compute_features(df, inst_df, margin_df, futures_oi_df, us_indices_df)

    stock_data_map = {}
    planner = RegimeTrainingPlan()
    
    for stock_id, group in feat_df.groupby("stock_id"):
        regime = KNOWN_REGIMES.get(stock_id, StockRegime.MID_CAP_TECH)
        feat_cols = [group[f].values if f in group.columns else np.zeros(len(group)) for f in FEATURE_NAMES]
        feat_tensor = np.nan_to_num(np.array(feat_cols, dtype=np.float32), nan=0.0, posinf=5.0, neginf=-5.0)

        horizon = planner.create_plan(stock_id, regime)["reward_horizon"]
        close = group["close"].values
        fwd_returns = np.zeros(len(close), dtype=np.float32)
        for i in range(len(close) - horizon):
            fwd_returns[i] = (close[i + horizon] - close[i]) / (close[i] + 1e-6)

        stock_data_map[stock_id] = {
            "feat": feat_tensor,
            "returns": fwd_returns,
            "regime_plan": planner.create_plan(stock_id, regime),
        }

    print("\n--- 步驟 3: GRPO 訓練 ---")
    try:
        import torch
        if not torch.cuda.is_available() or os.environ.get("GRPO_FORCE_CPU", "0") == "1":
            print("\n  [INFO] CPU 模式 → 只訓練 MID_CAP_TECH (單 regime)")
            stock_data_map = {k: v for k, v in stock_data_map.items() if v.get("regime_plan", {}).get("regime", StockRegime.MID_CAP_TECH) == StockRegime.MID_CAP_TECH}
    except ImportError: pass

    trainer = GRPOAlphaTrainer(GRPOConfig.auto_detect())
    results = trainer.train_all_regimes(stock_data_map)

    print("\n" + "=" * 60 + "\n  訓練完成!\n" + "=" * 60)
    final_formulas = {}
    for stock_id, res in results.items():
        regime = res['regime']
        if regime not in final_formulas: final_formulas[regime] = res

    regime_strategies = {}
    for regime, res in final_formulas.items():
        print(f"  Regime [{regime}]: train_IC={res.get('train_ic', 0):.4f} val_IC={res.get('val_ic', 0):.4f} → {res.get('decoded_formula', 'N/A')}")
        regime_strategies[regime] = {
            "formula_tokens": res.get("best_formula", []),
            "formula_str": res.get("decoded_formula", "N/A"),
            "train_ic": float(res.get("train_ic", 0)),
            "val_ic": float(res.get("val_ic", 0)),
            "best_reward": float(res.get("best_reward", 0)),
            "n_steps": res.get("n_steps", 0),
        }

    output = {
        "regimes": regime_strategies,
        "metadata": {
            "vocab_size": VOCAB_SIZE,
            "n_features": N_FEATURES,
            "version": "Anti_Collapse_T4",
            "n_stocks": len(stock_data_map),
            "accelerator": "NvidiaTeslaT4"
        },
    }

    with open("best_strategy_per_regime.json", "w") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
        
    report = {
        "version": "Anti_Collapse_T4",
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": {
            "device": trainer.config.device,
            "group_size": trainer.config.group_size,
            "train_steps": trainer.config.train_steps,
            "lr": trainer.config.lr,
        },
        "regime_results": {
            regime: {
                "formula": res.get("decoded_formula", "N/A"),
                "train_ic": float(res.get("train_ic", 0)),
                "val_ic": float(res.get("val_ic", 0)),
            } for regime, res in final_formulas.items()
        },
    }
    with open("training_report.json", "w") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

if __name__ == "__main__":
    main()

Data source import complete.
 台股 GRPO Regime-Aware 因子訓練 - Anti-Collapse (T4 Optimized)
  GPU Detected: Tesla T4 (15.6 GB), sm_75
  PyTorch version: 2.10.0+cu128
  [INFO] 成功匹配 Tesla T4 GPU，效能最佳化已啟用。
  [INFO] CUDA probe: PASS → GPU 訓練模式啟動

--- 步驟 1: 載入真實數據 (/kaggle/input) ---
  [adapt_finmind] 找到 5 個 CSV: ['price_ohlcv.csv', 'margin.csv', 'us_indices.csv', 'futures_oi.csv', 'inst_flow.csv']
  載入完成: 24385 筆, 20 檔股票

--- 步驟 2: 特徵工程 ---

--- 步驟 3: GRPO 訓練 ---

  訓練 regime=traditional (3 檔)
  [GRPO] regime=traditional, G=32, device=cuda, steps=12000, lr=0.0005
  step     0: loss=-4.0120 mean_r=-0.091 best_r=0.256 temp=2.00
               [Stats] with_ops=32/32 avg_ops=4.44 avg_len=8.0
               best_ops=2 best_len=5 val_ic_best=0.0987 best_composite=0.0810
               [Reward] ic=0.377 cplx=0.020 simp=0.000
  step   200: loss=-6.0407 mean_r=0.178 best_r=0.380 temp=1.97
               [Stats] with_ops=32/32 avg_ops=4.78 avg_len=8.2
               best_ops=5 best_len=9 val_ic_best=0.14

KeyboardInterrupt: 